# Bororo Synthetic Speech Generation

This notebook generates a controlled synthetic-speech dataset for Bororo, a low-resource Indigenous language, using Meta's MMS-TTS model. It creates clean and noisy variants together with a manifest that can be evaluated by the companion ASR benchmark notebook.

> **Data note:** the source dictionary is not redistributed because its licensing terms were not included with the course files. Place an authorized dictionary export at `data/boe-dict-20240805.json` before running.

# Bororo Speech Synthesis and ASR Benchmark

## Overview

This notebook implements a complete pipeline for **synthetic speech generation and evaluation** in the **Bororo language**, a low-resource indigenous language.

The workflow is divided into two main phases:

1. **Speech synthesis (TTS)** using Meta’s **MMS-TTS Bororo model**
2. **Automatic speech recognition (ASR) benchmarking** using Whisper

The goal is to generate a controlled and reproducible audio dataset and evaluate how well an ASR system can recover the original linguistic content from synthesized (and noisy) speech.

---

## Input Data

The notebook takes as input a **Bororo–Portuguese dictionary** (`boe-dict-20240805.json`), structured as:

- **keys**: Bororo words or multi-word expressions  
- **values**: Portuguese glosses or definitions  

Only the **Bororo terms (dictionary keys)** are used for speech synthesis. The Portuguese glosses are stored as metadata and are not synthesized.

---

## Speech Synthesis

For each Bororo term in the dictionary:

- Speech is synthesized once using the **MMS-TTS Bororo model**
- From this clean synthetic signal, **multiple noisy variants** are generated by adding controlled white noise

In [ ]:
# Installa le librerie necessarie (una volta sola)
!pip -q install "transformers>=4.33" accelerate soundfile

In [ ]:
# Import principali
import json, re, math, os
from pathlib import Path

import numpy as np
import torch
import soundfile as sf

# Path to an authorized Bororo–Portuguese dictionary export
BOE_DICT_PATH = Path("data/boe-dict-20240805.json")

# Cartella output
OUT_DIR = Path("artifacts/mms_bororo_audio")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("BOE_DICT_PATH:", BOE_DICT_PATH)
print("OUT_DIR:", OUT_DIR)

In [ ]:
# Carica il dizionario: {bororo_term: portuguese_gloss}
if not BOE_DICT_PATH.exists():
    raise FileNotFoundError(
        f"Non trovo {BOE_DICT_PATH}. "
        "Place an authorized dictionary export in data/ or modify BOE_DICT_PATH."
    )

with BOE_DICT_PATH.open("r", encoding="utf-8") as f:
    boe_dict = json.load(f)

if not isinstance(boe_dict, dict):
    raise ValueError("Il file non è un JSON object (dict).")

# Le “parole bororo” sono le CHIAVI
bororo_terms = [k.strip() for k in boe_dict.keys() if str(k).strip()]
bororo_terms = sorted(set(bororo_terms))

print("Numero di termini bororo trovati:", len(bororo_terms))
print("Esempi:", bororo_terms[:10])

In [ ]:
from transformers import VitsModel, AutoTokenizer

MODEL_ID = "facebook/mms-tts-bor"  # Borôro (ISO 639-3: bor)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = VitsModel.from_pretrained(MODEL_ID).to(device)
model.eval()

SAMPLE_RATE = int(model.config.sampling_rate)
print("Sample rate:", SAMPLE_RATE)

In [ ]:
def sanitize_filename(s: str, max_len: int = 80) -> str:
    """
    Converte una stringa in un nome file sicuro:
    - sostituisce spazi con _
    - rimuove caratteri strani
    - limita lunghezza
    """
    s = s.strip().lower()
    s = s.replace(" ", "_")
    s = re.sub(r"[^a-z0-9_\-]+", "", s)
    return s[:max_len] if len(s) > max_len else s

@torch.no_grad()
def synthesize_tts(text: str) -> np.ndarray:
    """
    Genera waveform (float32) con MMS TTS.
    """
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    out = model(**inputs).waveform  # shape: [1, T]
    wav = out[0].cpu().numpy().astype(np.float32)
    return wav

def add_white_noise(wav: np.ndarray, snr_db: float, rng: np.random.Generator) -> np.ndarray:
    """
    Aggiunge rumore bianco al segnale con un SNR target (in dB).

    snr_db alto = poco rumore (es. 30dB)
    snr_db basso = tanto rumore (es. 10dB)
    """
    if wav.size == 0:
        return wav

    # Potenza segnale
    sig_power = float(np.mean(wav**2) + 1e-12)

    # Potenza rumore richiesta per ottenere snr_db:
    # snr_db = 10*log10(sig_power / noise_power)  -> noise_power = sig_power / 10^(snr_db/10)
    noise_power = sig_power / (10 ** (snr_db / 10.0))

    noise = rng.normal(0.0, 1.0, size=wav.shape).astype(np.float32)
    noise = noise * math.sqrt(noise_power / (float(np.mean(noise**2)) + 1e-12))

    mixed = wav + noise

    # Clipping soft per stare in [-1, 1] senza distorsioni eccessive
    mixed = np.clip(mixed, -1.0, 1.0).astype(np.float32)
    return mixed

def save_wav(path: Path, wav: np.ndarray, sr: int):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), wav, sr)

In [ ]:
import csv
from tqdm import tqdm

# Parametri principali
N_PER_TERM = 50                 # <- richiesto: 50 audio per parola
SNR_DB = 20.0                   # livello rumore: 20dB è “realistico ma ancora intelligibile”
SEED = 1234                     # per riproducibilità
LIMIT_TERMS = None              # es: 20 per provare veloce; None = tutti

rng = np.random.default_rng(SEED)

# Output manifest (utile per benchmark Whisper: reference transcript = parola bororo)
MANIFEST_CSV = OUT_DIR / "manifest.csv"

terms_to_use = bororo_terms[:LIMIT_TERMS] if LIMIT_TERMS else bororo_terms

rows = []
total = len(terms_to_use) * N_PER_TERM
print(f"Genero {N_PER_TERM} audio per ciascuno dei {len(terms_to_use)} termini => {total} file...")

for term in tqdm(terms_to_use, desc="Termini"):
    safe_term = sanitize_filename(term)
    term_dir = OUT_DIR / safe_term
    term_dir.mkdir(parents=True, exist_ok=True)  # ✅ crea la cartella del termine
    gloss_pt = str(boe_dict.get(term, ""))       # definizione in portoghese (opzionale)

    # ✅ 1) TTS UNA SOLA VOLTA per termine (super accelerazione)
    wav_clean = synthesize_tts(term)

    for i in range(N_PER_TERM):
        # ✅ 2) Aggiungi rumore bianco (veloce) -> 50 varianti diverse
        noisy = add_white_noise(wav_clean, snr_db=SNR_DB, rng=rng)

        # 3) Salva
        out_path = term_dir / f"{safe_term}__{i:03d}.wav"
        save_wav(out_path, noisy, SAMPLE_RATE)

        # 4) Riga manifest
        rows.append({
            "audio": str(out_path),
            "transcript": term,       # reference (bororo)
            "gloss_pt": gloss_pt,     # opzionale
            "snr_db": SNR_DB
        })

# Scrivi manifest CSV
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["audio", "transcript", "gloss_pt", "snr_db"])
    writer.writeheader()
    writer.writerows(rows)

print("Fatto!")
print("Manifest:", MANIFEST_CSV)
print("Esempio cartella output:", OUT_DIR)

In [ ]:
import shutil

ZIP_PATH = Path("artifacts/mms_bororo_audio.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", root_dir=OUT_DIR)
print("Creato:", ZIP_PATH)